# Evolving a quantum compiler pass — search here, model on a GPU

The orchestrator — evolution loop, program database, evaluator — stays on **this**
instance, which can be a free CPU one. The model runs on an on-demand **GPU** that
gets launched, served, tunnelled back to `localhost`, and terminated for you.

Compared to `02_local_gpu_quickstart.ipynb`, which needs you to already be *on* a
GPU instance, this one is the cross-machine version: cheap orchestrator, expensive
worker, and the worker only exists while it is being used.

> **This spends credits** — an on-demand GPU is billed by the minute while it runs.
> `gpu-l4` is about $0.49/hour; `gpu-l40s`, big enough for a 14B model, about $2.28.

## How the machine is prevented from outliving you

Four independent stops, because a forgotten GPU is the most expensive mistake
available here:

| stop | where it runs | covers |
|---|---|---|
| `max_session_minutes` | qBraid server | this notebook dying, the kernel being killed, you closing the tab |
| `auto_stop_idle_minutes` | qBraid server | the run finishing but nothing using the GPU |
| `__exit__` | this process | normal completion, and any exception |
| `atexit` + orphan sweep | this process | a failure *between* provisioning and getting an instance id |

Only the first two survive this machine going away, which is exactly when you
would otherwise keep paying.

In [ ]:
import os
import pathlib
import sys

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("orchestrating from:", ROOT)

## 1. What is running right now?

Check before you start, so you know what you added — and check again at the end.

In [ ]:
from qbraid_core.services.compute.client import ComputeClient

client = ComputeClient()
before = client.list_bma_instances()
print(f"{len(before)} on-demand instance(s) running")
for instance in before:
    print(" ", instance.instance_id, instance.status)

## 2. Pick a GPU

Model size drives this more than anything else. ShinkaEvolve asks the model for
exactly-matching SEARCH/REPLACE diffs, and below ~14B the run tends to stall on
unparseable patches rather than fail cleanly — see `docs/CHOOSING_A_MODEL.md`.

| profile | VRAM | $/hour | comfortable at bf16 |
|---|---|---|---|
| `gpu-l4` | 24 GB | 0.49 | 7B |
| `gpu-l40s` | 48 GB | 2.28 | 14B |
| `gpu-a100-sxm` | 80 GB | 2.49 | 32B |
| `gpu-h100-sxm` | 80 GB | 5.37 | 32B, faster |

`qbraid compute list` has the current rates and what has capacity.

In [ ]:
PROFILE = "gpu-l40s"
MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
GENERATIONS = 40

## 3. Evolve

One cell. It launches the GPU, sets the guardrails, installs and starts vLLM,
tunnels port 8000 back here, runs the search against it, and terminates the
instance — including if this cell raises or you interrupt it.

The first run spends several minutes downloading weights before anything appears
to happen. Progress is printed as it goes.

In [ ]:
import subprocess

from qbraid_remote_gpu import RemoteGPUEndpoint

with RemoteGPUEndpoint(
    profile=PROFILE,
    model=MODEL,
    max_session_minutes=120,     # hard server-side ceiling
    auto_stop_idle_minutes=20,
) as gpu:
    print("endpoint:", gpu.base_url)
    subprocess.run(
        [
            sys.executable, "run_evolution.py",
            "--endpoint", "local",
            "--base-url", gpu.base_url,
            "--model", gpu.model,
            "--generations", str(GENERATIONS),
            "--results-dir", "results/remote_gpu",
        ],
        check=False,
    )

## 4. Confirm the GPU is gone

This should show exactly what it showed in step 1. If it does not, terminate by
hand — `qbraid compute instances terminate <id> --yes`.

In [ ]:
after = client.list_bma_instances()
print(f"{len(after)} on-demand instance(s) running")
for instance in after:
    print(" ", instance.instance_id, instance.status)
assert len(after) <= len(before), "something is still running -- terminate it"
print("\nclean")

## 5. Results

In [ ]:
import sqlite3

import pandas as pd

rows = sqlite3.connect("results/remote_gpu/programs.sqlite").execute(
    "SELECT generation, combined_score, correct FROM programs ORDER BY generation"
).fetchall()
df = pd.DataFrame(rows, columns=["generation", "score", "correct"])
rate = df.correct.mean()
print(f"{int(df.correct.sum())}/{len(df)} candidates scored ({rate:.0%})")
if rate < 0.4:
    print("LOW -- the model is likely too small for the diff protocol; size up.")
print(f"seed {df.score.iloc[0]:.4f}  ->  best {df.score.max():.4f}")

In [ ]:
import matplotlib.pyplot as plt

ok = df[df.correct == 1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(ok.generation, ok.score, s=24, label="candidate")
ax.plot(ok.generation, ok.score.cummax(), lw=2, color="crimson", label="best so far")
ax.axhline(1.0, ls="--", c="gray", lw=1, label="identity layout")
ax.set_xlabel("generation")
ax.set_ylabel("mean speedup vs identity layout")
ax.set_title(f"Evolution on {MODEL.split('/')[-1]}, served on {PROFILE}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
!python task/evaluate.py \
    --program_path results/remote_gpu/best/main.py \
    --results_dir /tmp/verify

## Running it from the shell instead

The same thing, without a notebook:

```bash
python qbraid_remote_gpu.py --profile gpu-l40s \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --generations 40
```

Add `--serve-only` to bring the endpoint up and hold it, if you would rather drive
the search yourself. `--keep-alive` leaves the instance running afterwards; the
hard session cap still applies.